In [1]:
import pandas as pd
import os
import numpy as np

In [2]:
myDF2=pd.read_csv("/LeeLab/HPRC/chromosomeY/Data/DAZ_Mappings/DAZNames_PSV.csv").set_index("Unnamed: 0")
myDF2['end']=[int(x.split("-")[1]) for x in myDF2.index]
myDF2['sample']=[x.split("_")[0] for x in myDF2['contig']]
goodDAZ = [x.split("_")[0] for x in set(myDF2['contig'])]

In [3]:
for x in goodDAZ:
    if 'HG01099' in x:
        print(x)

HG01099


In [4]:
len(goodDAZ)

100

In [5]:
dazDict={}
directory='/LeeLab/HPRC/chromosomeY/Data/geneAnnotation/CombinedFiles/'
for file in os.listdir(directory):
    sample = file.split("_")[0]
    if sample in goodDAZ:
        if sample == 'HG01099':
            print('YES')

        df = pd.read_csv(directory+file)
        df2 = df[(df['contig']==sample+"_chrY") & (df['gene_name'].str.contains("DAZ")) & (df['RM_Amplicon_Caller']=='CALLED')].copy()
        for gene, exons in zip(df2['RM_Amplicon_gene'], df2['RM_Amplicon_DAZrepeat_Pattern']):
            if gene in dazDict:
                if gene == 'DAZ1':
                    print(gene, len(exons), exons)
                    print(sample)
                dazDict[gene].append(len(exons))
            else:
                
                dazDict[gene]=[len(exons)]

DAZ1 9 ZYXEFEDCB
HG01358
DAZ1 9 ZYXEFEDCB
NA18608
DAZ1 14 BCDEFEXYYYYYYZ
HG03732
DAZ1 9 ZYXEFEDCB
HG02391
DAZ1 9 BCDEFEXYZ
HG03471
DAZ1 9 BCDEFEXYZ
NA20850
DAZ1 9 ZYXEFEDCB
NA19239
DAZ1 9 ZXXEFEDCB
NA18522
DAZ1 9 BCDEFEXYZ
NA19705
DAZ1 9 ZYXEFEDCB
HG02953
DAZ1 11 BCDEFEFEXYZ
HG02040
DAZ1 9 ZYXEFEDCB
200084
DAZ1 9 ZYXEFEDCB
NA19700
DAZ1 9 ZYXEFEDCB
HG04199
DAZ1 9 BCDEFEXYZ
HG01457
DAZ1 9 ZYXEFEDCB
NA20346
DAZ1 9 ZYXEFEDCB
NA19443
DAZ1 9 ZYXEFEDCB
HG02965
DAZ1 10 ZYYXEFEDCB
NA18612
DAZ1 9 BCDEFEXYZ
HG01952
DAZ1 9 ZYXEFEDCB
HG00512
DAZ1 9 ZYXEFEDCB
NA19331
DAZ1 11 ZYXEFEDEDCB
HG01109
DAZ1 9 ZYXEFEDCB
HG00621
DAZ1 9 ZYXEFEDCB
HG00096
DAZ1 10 BCDEFEXYYZ
HG02027
DAZ1 9 BCDEFEXYZ
HG01258
DAZ1 11 ZYXEFEDEDCB
HG01074
DAZ1 9 ZYXEFEDCB
HG00358
DAZ1 9 ZYXEFEDCB
HG01943
DAZ1 9 ZYXEFEDCB
HG02011
DAZ1 9 BCDEFEXYZ
NA20509
DAZ1 9 ZYXEFEDCB
HG03130
DAZ1 9 ZYXEFEDCB
HG00126
DAZ1 9 ZYXEFEDCB
NA12886
DAZ1 10 BCDEFEXYYZ
HG02083
DAZ1 14 BCDEFEXYYYYYYZ
HG03017
DAZ1 9 ZYXEFEDCB
HG01505
DAZ1 9 ZYXEFEDCB
HG02572

In [6]:
for daz in dazDict:
    print(daz, min(dazDict[daz]), max(dazDict[daz]))

DAZ4 8 17
DAZ3 7 18
DAZ1 9 14
DAZ2 9 24
DAZ3/DAZ4 10 14


In [7]:
import pandas as pd

def extract_gff3_attributes(
    df,
    attribute_col=8,
    attributes_to_extract=None,
    prefix=None
):

    if attributes_to_extract is not None:
        attributes_to_extract = set(attributes_to_extract)

    def parse_attributes(attr_string):
        parsed = {}

        if pd.isna(attr_string):
            return parsed

        fields = str(attr_string).split(";")

        for field in fields:
            if "=" not in field:
                continue

            key, value = field.split("=", 1)

            if attributes_to_extract is None or key in attributes_to_extract:
                colname = f"{prefix}_{key}" if prefix else key
                parsed[colname] = value

        return parsed

    attr_df = df[attribute_col].apply(parse_attributes).apply(pd.Series)
    out_df = pd.concat([df, attr_df], axis=1)

    return out_df


In [8]:
gencode1 = pd.read_csv("/LeeLab/HPRC/chromosomeY/Information/GENCODE/gencode.v49.annotation.gff3", comment='#',sep='\t', header=None)
gencodechrY = extract_gff3_attributes(gencode1[(gencode1[0]=='chrY') & (gencode1[2]=='gene')])
gencodechrX = extract_gff3_attributes(gencode1[(gencode1[0]=='chrX')& (gencode1[2]=='gene')])
allGencode = extract_gff3_attributes(gencode1[(gencode1[2]=='gene')].copy())
geneChromosomes ={x:y for x,y in zip(allGencode['gene_name'],allGencode[0])}

In [9]:
sampleDict={}
weirdDF = pd.read_csv("/LeeLab/HPRC/chromosomeY/Data/geneAnnotation/nonY_GeneList.csv")
for row in weirdDF.index:
    sample = weirdDF.at[row,'sample']
    if sample in sampleDict:
        sampleDict[sample].append(weirdDF.at[row,'gene'])
    else:
        sampleDict[sample]=[weirdDF.at[row,'gene']]

In [10]:
def _attr_get(attrs: str, key: str, default=None):
    if not isinstance(attrs, str):
        return default
    needle = key + "="
    if needle not in attrs:
        return default
    return attrs.split(needle, 1)[1].split(";", 1)[0]

In [11]:
geneHitDict={}
directory = '/LeeLab/HPRC/chromosomeY/Data/AmpliconicGenes/Feyza_gffFiles/'
for file in os.listdir(directory):
    if '.DS_Store' in file:
        continue
    else:
        sample = file.split("_")[0]

        if sample in set(weirdDF['sample']):
            geneHitDict[sample]={}
            df = pd.read_csv(os.path.join(directory, file), sep='\t', comment='#', header=None)
            df2 = df[df[2] == "gene"].copy()
            attrs = df2[8].astype(str)
            df2["GeneID"]      = attrs.map(lambda x: _attr_get(x, "ID"))
            df2["GeneName"]    = attrs.map(lambda x: _attr_get(x, "gene_name"))
            df2["GeneBiotype"] = attrs.map(lambda x: _attr_get(x, "gene_biotype"))
            df2["coverage"]    = pd.to_numeric(attrs.map(lambda x: _attr_get(x, "coverage")), errors="coerce")
            df2["sequence_ID"] = pd.to_numeric(attrs.map(lambda x: _attr_get(x, "sequence_ID")), errors="coerce")
            df2['chromosome']=[geneChromosomes[x] if x in geneChromosomes else x for x in df2["GeneName"]]

            for row in df2.index:
                gene = str(df2.at[row,'GeneName'])+"_"+str(df2.at[row,'chromosome'])
                geneHitDict[sample][gene]=str(df2.at[row,0])+":"+str(df2.at[row,3])+"-"+str(df2.at[row,4])
        else:
            continue

In [12]:
weirdDF = pd.read_csv("/LeeLab/HPRC/chromosomeY/Data/geneAnnotation/nonY_GeneList.csv")

In [13]:
weirdDF['Location']='NONE'
for row, sample, gene in zip(weirdDF.index, weirdDF['sample'], weirdDF['gene']):
    weirdDF.at[row,'Location']=geneHitDict[sample][gene]

## Input seq classes

In [14]:
import os
import pandas as pd

def add_overlap_names_from_testdf(
    wDF2: pd.DataFrame,
    testDF: pd.DataFrame,
    location_col: str = "Location",
    test_seq_col: str = "seq",
    test_start_col: str = "start",
    test_end_col: str = "end",
    test_name_col: str = "name",
) -> pd.DataFrame:
    w = wDF2.copy()
    w["orig_index"] = w.index   
    t = testDF.copy()

    w[["w_contig", "coords"]] = w[location_col].str.split(":", n=1, expand=True)
    w[["w_start", "w_end"]] = w["coords"].str.split("-", n=1, expand=True)
    w["w_start"] = pd.to_numeric(w["w_start"], errors="coerce")
    w["w_end"] = pd.to_numeric(w["w_end"], errors="coerce")
    w = w.drop(columns=["coords"])

    t[test_start_col] = pd.to_numeric(t[test_start_col], errors="coerce")
    t[test_end_col] = pd.to_numeric(t[test_end_col], errors="coerce")

    w = w.dropna(subset=["w_contig", "w_start", "w_end"])
    t = t.dropna(subset=[test_seq_col, test_start_col, test_end_col])

    merged = w.merge(
        t,
        left_on="w_contig",
        right_on=test_seq_col,
        how="left"
    )

    overlaps = merged[
        (merged["w_start"] <= merged[test_end_col]) &
        (merged["w_end"] >= merged[test_start_col])
    ].copy()

    overlap_df = (
        overlaps.groupby("orig_index")[test_name_col]
        .agg(lambda x: list(pd.unique(x.dropna())))
        .reset_index()
        .rename(columns={test_name_col: "overlap_names"})
    )

    out = w.merge(overlap_df, on="orig_index", how="left")

    out["overlap_names"] = out["overlap_names"].apply(
        lambda x: x if isinstance(x, list) else []
    )
    out["overlap_name"] = out["overlap_names"].apply(
        lambda x: ";".join(map(str, x)) if x else None
    )

    return out[list(wDF2.columns) + ["orig_index", "overlap_names", "overlap_name"]]

In [15]:
all_results = []

for sample in set(weirdDF["sample"]):
    bed_path = f"/LeeLab/HPRC/chromosomeY/QC/t2tv2/{sample}.t2tv2.chrY-regions.err-struct-base.bed"

    if not os.path.exists(bed_path):
        continue

    wDF2 = weirdDF[weirdDF["sample"] == sample].copy()

    testDF = pd.read_csv(
        bed_path,
        sep="\t",
        header=None,
        names=["seq", "start", "end", "name", "score", "strand"]
    )

    sample_result = add_overlap_names_from_testdf(wDF2, testDF)
    all_results.append(sample_result[["orig_index", "overlap_names", "overlap_name"]])

overlap_results = pd.concat(all_results, ignore_index=True)

weirdDF2 = weirdDF.copy()
weirdDF2["orig_index"] = weirdDF2.index

weirdDF2 = weirdDF2.merge(
    overlap_results,
    on="orig_index",
    how="left"
)

weirdDF2["overlap_names"] = weirdDF2["overlap_names"].apply(
    lambda x: x if isinstance(x, list) else []
)

weirdDF2 = weirdDF2.drop(columns=["orig_index"])

In [16]:
len(set(weirdDF2['gene']))

21

In [17]:
import os
import pandas as pd

def parse_location(loc):
    """
    Parse strings like:
    HG01433_chrY_random0000515:1131175-1139207
    HG01433_chrY:64061045-64075754

    Returns:
        contig, start_1based, end_1based
    """
    contig, coords = loc.split(":")
    start, end = coords.split("-")
    return contig, int(start), int(end)


all_overlap_rows = []

for sample in weirdDF2["sample"].unique():
    tempGeneDF = weirdDF2[weirdDF2["sample"] == sample].copy()

    segdup_path = f"/LeeLab/HPRC/chromosomeY/Data/segdups/{sample}.hap1.SDs.bed"
    segDupDF = pd.read_csv(segdup_path, sep="\t")

    parsed = tempGeneDF["Location"].apply(parse_location)
    tempGeneDF["gene_contig"] = parsed.str[0]
    tempGeneDF["gene_start_1based"] = parsed.str[1]
    tempGeneDF["gene_end_1based"] = parsed.str[2]

    tempGeneDF["gene_start"] = tempGeneDF["gene_start_1based"] - 1
    tempGeneDF["gene_end"] = tempGeneDF["gene_end_1based"]

    segDupDF = segDupDF.rename(columns={"#chr1": "segdup_contig"})

    sample_hits = []

    for _, gene_row in tempGeneDF.iterrows():
        overlaps = segDupDF[
            (segDupDF["segdup_contig"] == gene_row["gene_contig"]) &
            (gene_row["gene_start"] < segDupDF["end1"]) &
            (gene_row["gene_end"] > segDupDF["start1"])
        ].copy()

        if len(overlaps) > 0:
            overlaps["sample"] = sample
            overlaps["gene"] = gene_row["gene"]
            overlaps["Location"] = gene_row["Location"]
            overlaps["caller"] = gene_row["caller"]
            overlaps["biotype"] = gene_row["biotype"]
            overlaps["overlap_name"] = gene_row["overlap_name"]
            sample_hits.append(overlaps)

    if sample_hits:
        sample_hits = pd.concat(sample_hits, ignore_index=True)
        all_overlap_rows.append(sample_hits)

finalOverlapDF = pd.concat(all_overlap_rows, ignore_index=True) if all_overlap_rows else pd.DataFrame()

In [18]:
import os
import pandas as pd

def parse_location(loc):
    contig, coords = loc.split(":")
    start, end = coords.split("-")
    return contig, int(start), int(end)

results = []

for sample in weirdDF2["sample"].unique():
    tempGeneDF = weirdDF2[weirdDF2["sample"] == sample].copy()
    segdup_path = f"/LeeLab/HPRC/chromosomeY/Data/segdups/{sample}.hap1.SDs.bed"
    segDupDF = pd.read_csv(segdup_path, sep="\t").rename(columns={"#chr1": "segdup_contig"})

    parsed = tempGeneDF["Location"].apply(parse_location)
    tempGeneDF["gene_contig"] = parsed.str[0]
    tempGeneDF["gene_start"] = parsed.str[1] - 1   # convert to BED-like
    tempGeneDF["gene_end"] = parsed.str[2]

    has_overlap = []
    overlap_ids = []

    for _, row in tempGeneDF.iterrows():
        overlaps = segDupDF[
            (segDupDF["segdup_contig"] == row["gene_contig"]) &
            (row["gene_start"] < segDupDF["end1"]) &
            (row["gene_end"] > segDupDF["start1"])
        ]

        has_overlap.append(len(overlaps) > 0)
        overlap_ids.append(";".join(overlaps["name"].astype(str).tolist()) if len(overlaps) > 0 else None)

    tempGeneDF["segdup_overlap"] = has_overlap
    tempGeneDF["segdup_names"] = overlap_ids

    results.append(tempGeneDF)

finalDFSimple = pd.concat(results, ignore_index=True)

In [39]:
#finalDFSimple.drop(columns=['Unnamed: 0','index']).to_csv("/LeeLab/HPRC/chromosomeY/Data/AmpliconicGenes/nonY_geneAnnotation_04092026.csv")

In [20]:
import collections
collections.Counter(finalDFSimple['segdup_overlap'])

Counter({False: 454, True: 343})

In [111]:
#343/797

0.43036386449184444

In [21]:
finalDFSimple[(finalDFSimple['segdup_overlap']==True) & (finalDFSimple['gene']=='SPRY3_chrX')]

,Unnamed: 0,biotype,sample,contig,index,caller,gene,chrom,Haplotype,Location,overlap_names,overlap_name,gene_contig,gene_start,gene_end,segdup_overlap,segdup_names
142,377,protein_coding,HG02717,HG02717_chrY,1364,liftoff,SPRY3_chrX,chrX,E1b1a1a1a1c2c3a2a,HG02717_chrY:54740125-54754832,[PAR2],PAR2,HG02717_chrY,54740124,54754832,True,pat-0000667:152989422-153326267


In [22]:
''''AZFb;AMPL7;AZFc;P1;yellow': 77,
         'AZFc;AZFb;AMPL7;P1;yellow': 32,
         'AZFb;AZFc;AMPL7;P1;yellow': 27,
         'AZFc;yellow;P1;AZFb;AMPL7': 22,
         'AMPL7;AZFb;AZFc;P1;yellow': 16,
         'AZFc;yellow;AZFb;AMPL7': 12,
         'AZFc;AZFb;yellow': 9,
         'AMPL7;AZFc;AZFb;P1;yellow': 6,
         'yellow': 4,
         'yellow;AZFb;AZFc;AMPL7': 4,
         'AZFb;AZFc;AMPL7;yellow': 3,
         'AZFb;AMPL7;AZFc;yellow': 3,
         'yellow;P1;AZFb;AMPL7;AZFc': 3,
         'AZFc;AMPL7;AZFb;P1;yellow': 3,
         'yellow;AMPL7;AZFb;AZFc;P1': 2,
         'AMPL7;AZFc;AZFb;yellow': 2,
         'AZFc;P1;yellow;AZFb;AMPL7': 2,
         'AMPL7;AZFc;yellow;P1;AZFb': 2,
'yellow;AMPL7;AZFb;AZFc': 1,
         'AZFb;AMPL7;AZFc;yellow;P1': 1,
AZFc;AZFb;AMPL7;yellow': 1,
         'P1;AMPL7;AZFb;AZFc;yellow': 1,
         'AZFc;yellow;P1;AZFb;AMPL7;ERRBASE': 1,
         'AZFb;AMPL7;P1;AZFc;yellow': 1,
         'yellow;AMPL7;AZFc;AZFb': 1,
         'AMPL7;AZFc;yellow;AZFb': 1,
         'AZFc;P1;AMPL7;AZFb;yellow': 1,
         'yellow;P1;AZFb;AZFc;AMPL7': 1}
         '''

"'AZFb;AMPL7;AZFc;P1;yellow': 77,\n         'AZFc;AZFb;AMPL7;P1;yellow': 32,\n         'AZFb;AZFc;AMPL7;P1;yellow': 27,\n         'AZFc;yellow;P1;AZFb;AMPL7': 22,\n         'AMPL7;AZFb;AZFc;P1;yellow': 16,\n         'AZFc;yellow;AZFb;AMPL7': 12,\n         'AZFc;AZFb;yellow': 9,\n         'AMPL7;AZFc;AZFb;P1;yellow': 6,\n         'yellow': 4,\n         'yellow;AZFb;AZFc;AMPL7': 4,\n         'AZFb;AZFc;AMPL7;yellow': 3,\n         'AZFb;AMPL7;AZFc;yellow': 3,\n         'yellow;P1;AZFb;AMPL7;AZFc': 3,\n         'AZFc;AMPL7;AZFb;P1;yellow': 3,\n         'yellow;AMPL7;AZFb;AZFc;P1': 2,\n         'AMPL7;AZFc;AZFb;yellow': 2,\n         'AZFc;P1;yellow;AZFb;AMPL7': 2,\n         'AMPL7;AZFc;yellow;P1;AZFb': 2,\n'yellow;AMPL7;AZFb;AZFc': 1,\n         'AZFb;AMPL7;AZFc;yellow;P1': 1,\nAZFc;AZFb;AMPL7;yellow': 1,\n         'P1;AMPL7;AZFb;AZFc;yellow': 1,\n         'AZFc;yellow;P1;AZFb;AMPL7;ERRBASE': 1,\n         'AZFb;AMPL7;P1;AZFc;yellow': 1,\n         'yellow;AMPL7;AZFc;AZFb': 1,\n         'AMPL7

In [23]:
#239/343

In [24]:
sum([y for x,y in collections.Counter(finalDFSimple[(finalDFSimple['segdup_overlap']==True) & (finalDFSimple['overlap_name'].str.contains("yellow"))]['overlap_name']).items()])

239

In [ ]:
#DDX11L5_chr9 Counter({True: 3})
#SEPTIN14P17_chr1 Counter({True: 1})
#UBE2Q2P6_chr15 Counter({True: 132})
#SEPTIN14P24_chr7 Counter({True: 4})
#SEPTIN14P2_chr2 Counter({True: 12})
#CICP16_chr4 Counter({True: 24})
#CICP19_chr19 Counter({True: 1})
#MIR6859-4_chr16 Counter({True: 37})
#RNU6-689P_chr14 Counter({True: 1})
#GYG2_chrX Counter({True: 1})
#SEPTIN14P19_chr19 Counter({True: 8})
#DDX11L9_chr15 Counter({True: 1})
#RPL41P1_chr20 Counter({True: 6})
#CICP9_chr10 Counter({True: 3})
#UBE2Q2P12_chr15 Counter({True: 22})
#MIR6859-1_chr1 Counter({True: 28})
#UBE2Q2P11_chr15 Counter({True: 26})

In [112]:
3+1+132+4+12+24+1+37+1+1+8+1+6+3+22+28+26

310

In [113]:
310/343

0.9037900874635568

In [ ]:
#SPRY3_chrX Counter({False: 138, True: 1}) PAR
#WASIR2_chr16 Counter({False: 37, True: 31}) PAR
#MIR4448_chr3 Counter({False: 139, True: 1}) XDR
#PRKX-AS1_chrX Counter({False: 140}) XDR

In [25]:
uniqueNameList = ['CICP',  'DDX11L', 'GYG2','MIR4448','MIR6859','RNU6-689P','RPL41P1','SEPTIN14P','SPRY3','UBE2Q2P','WASIR2','PRKX-AS1']
print(len(uniqueNameList))

12


In [26]:
set(weirdDF2['gene']) - set(finalOverlapDF['gene'])

{'PRKX-AS1_chrX'}

In [27]:
hapGenes=['CICP19_chr19', 'CICP9_chr10','DDX11L9_chr15','GYG2_chrX', 'RNU6-689P_chr14', 'SEPTIN14P17_chr1','SEPTIN14P2_chr2']

In [28]:
collections.Counter(weirdDF2['biotype'])

Counter({'pseudogene': 240,
         'lncRNA': 208,
         'miRNA': 205,
         'protein_coding': 140,
         'transcribed_pseudogene': 4})

In [95]:
#657/797

0.8243412797992472

In [29]:
nameList=[]
for gene, name in zip(weirdDF2['gene'], weirdDF2['overlap_name']):
    nameList.append(gene+"-"+';'.join(sorted(name.split(";"))))

In [30]:
TotalCount=0
for x,y in collections.Counter(nameList).items():
    if 'PAR' in x:
        print(x,y)
        TotalCount+=y
print(TotalCount)

MIR6859-1_chr1-PAR2 28
SPRY3_chrX-PAR2 134
WASIR2_chr16-PAR2 67
DDX11L5_chr9-PAR2 3
MIR6859-4_chr16-PAR2 36
DDX11L9_chr15-PAR2 1
SPRY3_chrX-ERRBASE;PAR2 4
GYG2_chrX-PAR1 1
WASIR2_chr16-DYZ2_Con;PAR2 1
MIR6859-4_chr16-DYZ2_Con;PAR2 1
SPRY3_chrX-DYZ2_Con;PAR2 1
277


In [31]:
TotalCount=0
for x,y in collections.Counter(nameList).items():
    if 'XDR' in x:
        print(x,y)
        TotalCount+=y
print(TotalCount)

PRKX-AS1_chrX-XDR2 137
MIR4448_chr3-AZFb;XDR7 138
MIR4448_chr3-XDR6 2
PRKX-AS1_chrX-ERRBASE;XDR2 3
280


In [32]:
TotalCount=0
for x,y in collections.Counter(nameList).items():
    if 'XDR' in x or 'PAR' in x:
        continue
    else:
        TotalCount+=y
        print(x,y)
print(TotalCount)

UBE2Q2P6_chr15-AMPL7;AZFb;AZFc;P1;yellow 108
CICP16_chr4-AMPL7;AZFb;AZFc;P1;yellow 18
UBE2Q2P6_chr15-AMPL7;AZFb;AZFc;yellow 18
CICP9_chr10-AMPL7;AZFb;AZFc;yellow 3
UBE2Q2P11_chr15-AMPL7;AZFb;AZFc;P1;yellow 26
RNU6-689P_chr14-UNASSIGNED 1
RPL41P1_chr20-AMPL7;AZFb;AZFc;P1;yellow 5
UBE2Q2P12_chr15-yellow 2
SEPTIN14P24_chr7-AMPL7;AZFb;AZFc;P1;yellow 4
UBE2Q2P12_chr15-AMPL7;AZFb;AZFc;P1;yellow 18
CICP19_chr19-AMPL7;AZFb;AZFc;P1;yellow 1
SEPTIN14P19_chr19-AMPL7;AZFb;AZFc;P1;yellow 6
SEPTIN14P2_chr2-AMPL7;AZFb;AZFc;P1;yellow 10
SEPTIN14P19_chr19-AMPL7;AZFb;AZFc;yellow 2
RPL41P1_chr20-AMPL7;AZFb;AZFc;yellow 1
SEPTIN14P2_chr2-AMPL7;AZFb;AZFc;yellow 2
UBE2Q2P6_chr15-AMPL7;AZFb;AZFc;ERRBASE;P1;yellow 1
SEPTIN14P17_chr1-AMPL7;AZFb;AZFc;P1;yellow 1
UBE2Q2P12_chr15-AMPL7;AZFb;AZFc;yellow 1
UBE2Q2P12_chr15-AZFb;AZFc;yellow 1
UBE2Q2P6_chr15-yellow 1
CICP16_chr4-yellow 1
CICP16_chr4-AZFb;AZFc;yellow 4
UBE2Q2P6_chr15-AZFb;AZFc;yellow 4
CICP16_chr4-AMPL7;AZFb;AZFc;yellow 1
240


In [33]:
import collections
myShortList=[]
for x,y in collections.Counter(nameList).items():
    for i in range(1,y+1):
        myShortList.append('-'.join(x.split("-")[:-1]))
collections.Counter(myShortList)

Counter({'PRKX-AS1_chrX': 140,
         'MIR4448_chr3': 140,
         'SPRY3_chrX': 139,
         'UBE2Q2P6_chr15': 132,
         'WASIR2_chr16': 68,
         'MIR6859-4_chr16': 37,
         'MIR6859-1_chr1': 28,
         'UBE2Q2P11_chr15': 26,
         'CICP16_chr4': 24,
         'UBE2Q2P12_chr15': 22,
         'SEPTIN14P2_chr2': 12,
         'SEPTIN14P19_chr19': 8,
         'RPL41P1_chr20': 6,
         'SEPTIN14P24_chr7': 4,
         'DDX11L5_chr9': 3,
         'CICP9_chr10': 3,
         'RNU6-689P_chr14': 1,
         'DDX11L9_chr15': 1,
         'CICP19_chr19': 1,
         'SEPTIN14P17_chr1': 1,
         'GYG2_chrX': 1})

In [34]:
blockDF = pd.read_csv("/LeeLab/HPRC/chromosomeY/Data/DavidPorubsky/AZFc_ColorBLock_Clusters_12162025.csv")